# 08 — DenseNet121 Resolution Screen (320 vs 384)

Mục tiêu: kiểm tra xem tăng resolution có cải thiện DenseNet121 trên **fold 0** hay không trước khi tốn chi phí 5-fold.

Giữ cố định:
- same locked split
- DenseNet121 ImageNet pretrained
- weighted cross-entropy
- AdamW + cosine LR
- mild affine + brightness/contrast augmentation
- positive class: Pneumonia = 1
- model selection bằng official P tại threshold 0.50

Thử:
- 320×320
- 384×384

Baseline fold-0 hiện tại:
- DenseNet121 @224 P@0.50 = 0.9851505
- single-fold tuned P = 0.9872839

Internal holdout 770 ảnh **không được dùng**.


In [ ]:
print('>>> CELL 1 START: imports/config', flush=True)
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'

from pathlib import Path
import json, time, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

SEED=2026
ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'screen_outputs'/'densenet_resolution'
OUT.mkdir(parents=True,exist_ok=True)

FOLD=0
EPOCHS=8
PATIENCE=3
LR=1e-4
WD=1e-4
PROGRESS_EVERY=50
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark=USE_AMP

CANDIDATES=[
    {'image_size':320,'batch_size':8,'grad_accum':1},
    {'image_size':384,'batch_size':4,'grad_accum':2},
]

REFERENCE_P_AT_05=0.9851505114783624
REFERENCE_TUNED_P=0.9872839166987125

print('PyTorch:',torch.__version__,flush=True)
print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Candidates:',CANDIDATES,flush=True)


In [ ]:
print('>>> CELL 2 START: load locked fold 0 split', flush=True)
m=pd.read_csv(MANIFEST)
dev=m[m.split=='dev'].copy().reset_index(drop=True)
holdout=m[m.split=='holdout'].copy().reset_index(drop=True)
train_df=dev[dev.cv_fold!=FOLD].copy().reset_index(drop=True)
val_df=dev[dev.cv_fold==FOLD].copy().reset_index(drop=True)

assert len(train_df)==3489
assert len(val_df)==873
assert len(holdout)==770
assert set(train_df.sha256).isdisjoint(set(val_df.sha256))
assert set(dev.sha256).isdisjoint(set(holdout.sha256))

print('Train:',len(train_df),'Val:',len(val_df),'SEALED holdout:',len(holdout),flush=True)


In [ ]:
print('>>> CELL 3 START: helpers', flush=True)

class PadToSquare:
    def __call__(self,img):
        w,h=img.size; s=max(w,h)
        l=(s-w)//2; r=s-w-l; t=(s-h)//2; b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]
L2I={'Normal':0,'Pneumonia':1}

def preload_frame(frame,size):
    pad=PadToSquare()
    arr=[]
    print(f'Preloading {len(frame)} images -> {size}x{size}...',flush=True)
    t0=time.time()
    for i,row in frame.iterrows():
        with Image.open(row.path) as im:
            im=im.convert('RGB')
            im=pad(im)
            im=im.resize((size,size),Image.Resampling.BILINEAR)
            arr.append(np.asarray(im,dtype=np.uint8))
        if (i+1)%500==0 or (i+1)==len(frame):
            print(f'  cached {i+1}/{len(frame)}',flush=True)
    out=np.stack(arr,axis=0)
    print(f'Cache RAM={out.nbytes/1024**3:.2f} GiB | {time.time()-t0:.0f}s',flush=True)
    return out

class CachedDataset(Dataset):
    def __init__(self,frame,cache,tf):
        self.df=frame.reset_index(drop=True)
        self.cache=cache
        self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        im=Image.fromarray(self.cache[i])
        return self.tf(im),L2I[row.label],row.filename

def make_transforms():
    train_tf=transforms.Compose([
        transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
        transforms.ColorJitter(brightness=0.08,contrast=0.08),
        transforms.ToTensor(),
        transforms.Normalize(MEAN,STD),
    ])
    val_tf=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN,STD),
    ])
    return train_tf,val_tf

def metric(y,prob,t=0.5,auc_value=None):
    y=np.asarray(y,int); prob=np.asarray(prob,float)
    pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y)
    se=tp/max(tp+fn,1)
    sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y,prob) if auc_value is None else float(auc_value)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),
            'Specificity':float(sp),'F1':float(f1),'AUROC':float(auc),
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'threshold':float(t)}

@torch.no_grad()
def predict(model,loader):
    model.eval(); ys=[]; ps=[]; names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),dim=1)[:,1]
        ys.extend(y.numpy()); ps.extend(p.cpu().numpy()); names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def make_model():
    w=models.DenseNet121_Weights.DEFAULT
    model=models.densenet121(weights=w)
    model.classifier=nn.Linear(model.classifier.in_features,2)
    return model.to(DEVICE)

def threshold_scan(y,p):
    auc=float(roc_auc_score(y,p))
    ts=np.round(np.arange(0.10,0.9001,0.005),3)
    df=pd.DataFrame([metric(y,p,t,auc_value=auc) for t in ts])
    best=df.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
    return df,best


## Fold-0 resolution screen

Mỗi resolution được train độc lập. Cache RAM được giải phóng giữa hai candidate.


In [ ]:
print('>>> CELL 5 START: train 320 and 384', flush=True)
results=[]
all_preds={}

for ci,cfg in enumerate(CANDIDATES,1):
    size=cfg['image_size']
    batch=cfg['batch_size']
    accum=cfg['grad_accum']

    print('\n'+'='*80,flush=True)
    print(f'CANDIDATE {ci}/{len(CANDIDATES)}: DenseNet121 @ {size} | batch={batch} accum={accum}',flush=True)
    print('='*80,flush=True)

    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if USE_AMP: torch.cuda.manual_seed_all(SEED)

    train_cache=preload_frame(train_df,size)
    val_cache=preload_frame(val_df,size)
    train_tf,val_tf=make_transforms()

    train_loader=DataLoader(CachedDataset(train_df,train_cache,train_tf),
                            batch_size=batch,shuffle=True,num_workers=0,pin_memory=False)
    val_loader=DataLoader(CachedDataset(val_df,val_cache,val_tf),
                          batch_size=batch,shuffle=False,num_workers=0,pin_memory=False)

    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

    model=make_model()
    counts=train_df.label.value_counts()
    n0=int(counts['Normal']); n1=int(counts['Pneumonia']); total=n0+n1
    cw=torch.tensor([total/(2*n0),total/(2*n1)],dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=cw)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best=-1.0; best_epoch=0; best_state=None; stale=0; hist=[]

    for epoch in range(1,EPOCHS+1):
        et0=time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_sum=0.0; seen=0

        for bi,(x,y,_) in enumerate(train_loader,1):
            x=x.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)

            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                logits=model(x)
                raw_loss=criterion(logits,y)
                loss=raw_loss/accum

            scaler.scale(loss).backward()
            if (bi%accum==0) or (bi==len(train_loader)):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_sum+=float(raw_loss.item())*len(x)
            seen+=len(x)

            if bi%PROGRESS_EVERY==0 or bi==len(train_loader):
                print(f'  {size}px epoch {epoch:02d} batch {bi}/{len(train_loader)} | loss={loss_sum/seen:.4f}',flush=True)

        scheduler.step()
        yv,pv,_=predict(model,val_loader)
        mm=metric(yv,pv,0.5)
        sec=time.time()-et0
        hist.append({'image_size':size,'epoch':epoch,'train_loss':loss_sum/seen,'seconds':sec,**mm})

        print(f"{size}px Epoch {epoch:02d} | P={mm['P']:.5f} | A={mm['Accuracy']:.5f} | "
              f"Se={mm['Sensitivity']:.5f} | Sp={mm['Specificity']:.5f} | "
              f"F1={mm['F1']:.5f} | AUROC={mm['AUROC']:.6f} | {sec:.0f}s",flush=True)

        if mm['P']>best+1e-6:
            best=mm['P']; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE:
                print(f'{size}px early stopping at epoch {epoch}',flush=True)
                break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,val_loader)
    fixed=metric(yv,pv,0.5)
    scan,brow=threshold_scan(yv,pv)
    bt=float(brow.threshold)
    tuned=metric(yv,pv,bt)

    pred=pd.DataFrame({'filename':names,'y_true':yv,'p_pneumonia':pv})
    pred.to_csv(OUT/f'densenet121_{size}_fold0_predictions.csv',index=False)
    pd.DataFrame(hist).to_csv(OUT/f'densenet121_{size}_fold0_history.csv',index=False)
    scan.to_csv(OUT/f'densenet121_{size}_fold0_threshold_scan.csv',index=False)
    torch.save({'architecture':'densenet121','image_size':size,'fold':0,
                'best_epoch':best_epoch,'state_dict':best_state},
               OUT/f'densenet121_{size}_fold0_best.pt')

    summary={
        'model':'DenseNet121','image_size':size,'fold':0,'best_epoch':best_epoch,
        'batch_size':batch,'grad_accum':accum,
        'metrics_at_0_5':fixed,'best_threshold':bt,'tuned_metrics':tuned,
        'holdout_used':False
    }
    with open(OUT/f'densenet121_{size}_fold0_summary.json','w',encoding='utf-8') as f:
        json.dump(summary,f,indent=2)

    results.append({
        'image_size':size,'best_epoch':best_epoch,
        'P_at_0_5':fixed['P'],'AUROC':fixed['AUROC'],
        'Sensitivity':fixed['Sensitivity'],'Specificity':fixed['Specificity'],
        'F1':fixed['F1'],'best_threshold':bt,'P_tuned_fold0':tuned['P'],
        'delta_vs_224_P_at_0_5':fixed['P']-REFERENCE_P_AT_05,
        'delta_vs_224_tuned':tuned['P']-REFERENCE_TUNED_P,
    })
    all_preds[size]=pred

    del model,optimizer,scheduler,scaler,best_state,train_loader,val_loader
    del train_cache,val_cache
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

print('>>> BOTH RESOLUTIONS FINISHED',flush=True)


In [ ]:
print('>>> CELL 6 START: comparison', flush=True)
table=pd.DataFrame(results).sort_values('P_at_0_5',ascending=False)
table.to_csv(OUT/'resolution_screen_summary.csv',index=False)

baseline=pd.DataFrame([{
    'image_size':224,
    'best_epoch':6,
    'P_at_0_5':REFERENCE_P_AT_05,
    'AUROC':0.9988618382699942,
    'Sensitivity':0.9800613496932515,
    'Specificity':0.9909502262443439,
    'F1':0.988399071925754,
    'best_threshold':0.205,
    'P_tuned_fold0':REFERENCE_TUNED_P,
    'delta_vs_224_P_at_0_5':0.0,
    'delta_vs_224_tuned':0.0,
}])

comparison=pd.concat([baseline,table],ignore_index=True).sort_values('P_at_0_5',ascending=False)
comparison.to_csv(OUT/'resolution_comparison_with_224.csv',index=False)
display(comparison)

winner=comparison.iloc[0]
print('\nBest by P@0.50:',winner[['image_size','P_at_0_5','P_tuned_fold0']].to_dict(),flush=True)

# Conservative rule: only promote a higher resolution if P@0.50 beats 224 baseline.
promote=table[table.P_at_0_5>REFERENCE_P_AT_05].sort_values('P_at_0_5',ascending=False)
if len(promote):
    print('PROMOTE TO 5-FOLD:',int(promote.iloc[0].image_size),'px',flush=True)
else:
    print('NO HIGHER RESOLUTION BEAT 224 AT P@0.50 -> keep 224 and move to preprocessing ablation.',flush=True)


## Decision rule

- Ưu tiên **P@0.50** để chọn candidate, vì threshold tuning trên một fold dễ overfit.
- Chỉ đưa 320/384 sang 5-fold nếu nó vượt baseline 224 đủ rõ.
- Nếu cả hai không thắng, giữ 224 và chuyển sang preprocessing / augmentation / loss ablation.
- Internal holdout vẫn niêm phong.
